# MMS Hausa VITS Fine-Tuning in Google Colab

This notebook clones `Mouhamadmm466/VITS_namu_fine_tuning`, prepares the local Hausa dataset, downloads the official `facebook/mms-tts-hau` full checkpoint, fine-tunes it with the original VITS training code, and synthesizes a test sample.

It is designed for a fresh Colab GPU runtime and assumes the repo contains the codebase plus the dataset files `metadata_wav.csv` and `wav/`.

## Workflow

1. Set runtime parameters and optional Google Drive output storage.
2. Clone your GitHub repo into the Colab session.
3. Install the minimal dependencies needed by this pipeline and build the VITS monotonic alignment extension.
4. Download the official MMS Hausa full-model checkpoint files.
5. Prepare the dataset with the repo's text normalization and pronunciation override pipeline.
6. Fine-tune the model.
7. Monitor logs and synthesize a test sentence.

### Important note

If you later decide not to store the `wav/` folder in GitHub, keep the notebook the same and simply copy `metadata_wav.csv` and `wav/` into the cloned repo path before running the dataset preparation cell.

In [ ]:
from pathlib import Path
import json
import shutil
import subprocess

REPO_URL = "https://github.com/Mouhamadmm466/VITS_namu_fine_tuning.git"
REPO_BRANCH = "main"

PROJECT_DIR = Path("/content/VITS_namu_fine_tuning")
VITS_DIR = Path("/content/vits")
MMS_DIR = Path("/content/mms_hau_full")
PREP_DIR = PROJECT_DIR / "prepared_mms_hau"

TRAINING_MAX_STEPS = 1500
TRAIN_BATCH_SIZE = 4
LEARNING_RATE = 1e-4
VAL_RATIO = 0.10
# 'dec,dp' trains only the decoder and duration predictor — correct for small datasets.
# Use 'all' only if you have hours of data.
TRAINABLE_MODULES = "dec,dp"

USE_DRIVE_FOR_OUTPUTS = True
DRIVE_RUNS_DIR = Path("/content/drive/MyDrive/vits_namu_runs/mms_hau_single_speaker")

TEST_SENTENCE = "sannu yaya kake yau"

def run(cmd, cwd=None):
    printable = " ".join(str(part) for part in cmd)
    print(f"$ {printable}")
    try:
        completed = subprocess.run(
            [str(part) for part in cmd],
            cwd=str(cwd) if cwd else None,
            check=True,
            text=True,
            capture_output=True,
        )
        if completed.stdout:
            print(completed.stdout)
        if completed.stderr:
            print(completed.stderr)
        return completed
    except subprocess.CalledProcessError as exc:
        print("Command failed.")
        print(f"Return code: {exc.returncode}")
        if exc.stdout:
            print("--- stdout ---")
            print(exc.stdout)
        if exc.stderr:
            print("--- stderr ---")
            print(exc.stderr)
        raise

print("Notebook configuration loaded.")
print(f"Repo: {REPO_URL}")
print(f"Training max steps: {TRAINING_MAX_STEPS}")
print(f"Batch size: {TRAIN_BATCH_SIZE}")
print(f"Learning rate: {LEARNING_RATE}")
print(f"Trainable modules: {TRAINABLE_MODULES}")

In [ ]:
if USE_DRIVE_FOR_OUTPUTS:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_RUNS_DIR.mkdir(parents=True, exist_ok=True)
    print(f"Training outputs will be written to {DRIVE_RUNS_DIR}")
else:
    print("Google Drive output storage is disabled.")

In [ ]:
if PROJECT_DIR.exists():
    shutil.rmtree(PROJECT_DIR)

run(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, PROJECT_DIR])

metadata_path = PROJECT_DIR / "metadata_wav.csv"
wav_dir = PROJECT_DIR / "wav"

if not metadata_path.exists():
    raise FileNotFoundError(f"Missing dataset file: {metadata_path}")
if not wav_dir.exists():
    raise FileNotFoundError(f"Missing dataset directory: {wav_dir}")

wav_count = len(list(wav_dir.glob("*.wav")))
print(f"Repo cloned to {PROJECT_DIR}")
print(f"Found metadata file: {metadata_path}")
print(f"Found {wav_count} WAV files in {wav_dir}")

In [ ]:
run(["pip", "install", "-q", "soundfile", "scipy", "tensorboard", "cython", "matplotlib", "librosa", "Unidecode", "phonemizer"])

if VITS_DIR.exists():
    shutil.rmtree(VITS_DIR)

run(["git", "clone", "https://github.com/jaywalnut310/vits.git", VITS_DIR])

# monotonic_align: create the nested package directory so the build finds the right output path.
nested_monotonic_dir = VITS_DIR / "monotonic_align" / "monotonic_align"
nested_monotonic_dir.mkdir(parents=True, exist_ok=True)
(nested_monotonic_dir / "__init__.py").touch()
run(["python", "setup.py", "build_ext", "--inplace"], cwd=VITS_DIR / "monotonic_align")

# ── Patch mel_processing.py ──────────────────────────────────────────────────
# PyTorch >= 1.8 requires return_complex=True on torch.stft for real inputs.
# The current VITS repo uses center=center and 1e-6 tolerance.
# We add return_complex=True and switch magnitude from .sum(-1) to .real/.imag.
mel_proc_path = VITS_DIR / "mel_processing.py"
mel_proc_text = mel_proc_path.read_text()
mel_proc_text = mel_proc_text.replace(
    "center=center, pad_mode='reflect', normalized=False, onesided=True)",
    "center=center, pad_mode='reflect', normalized=False, onesided=True, return_complex=True)",
)
mel_proc_text = mel_proc_text.replace(
    "spec = torch.sqrt(spec.pow(2).sum(-1) + 1e-6)",
    "spec = torch.sqrt(spec.real.pow(2) + spec.imag.pow(2) + 1e-6)",
)
mel_proc_path.write_text(mel_proc_text)
assert "return_complex=True" in mel_proc_path.read_text(), "mel_processing patch failed — stft line not found"
assert "spec.real.pow(2)" in mel_proc_path.read_text(), "mel_processing patch failed — magnitude line not found"
print("Patched mel_processing.py (return_complex=True).")

# ── Patch utils.py ───────────────────────────────────────────────────────────
# PyTorch >= 2.6 defaults torch.load to weights_only=True, which rejects the
# Python dicts (iteration, learning_rate, optimizer state) inside VITS checkpoints.
# Explicitly set weights_only=False so checkpoints load on all PyTorch versions.
utils_path = VITS_DIR / "utils.py"
utils_text = utils_path.read_text()
utils_text = utils_text.replace(
    "torch.load(checkpoint_path, map_location='cpu')",
    "torch.load(checkpoint_path, map_location='cpu', weights_only=False)",
)
utils_path.write_text(utils_text)
assert "weights_only=False" in utils_path.read_text(), "utils patch failed — torch.load line not found"
print("Patched utils.py (weights_only=False).")

print("All VITS patches applied successfully.")

In [ ]:
MMS_DIR.mkdir(parents=True, exist_ok=True)
mms_base_url = "https://huggingface.co/facebook/mms-tts/resolve/main/full_models/hau"

for filename in ["G_100000.pth", "D_100000.pth", "config.json", "vocab.txt"]:
    run(["wget", "-q", "-O", MMS_DIR / filename, f"{mms_base_url}/{filename}"])

print("Downloaded official MMS Hausa files:")
for path in sorted(MMS_DIR.iterdir()):
    print(f"- {path.name}")

In [ ]:
import csv
import importlib

for module_name in ["numpy", "soundfile", "scipy"]:
    importlib.import_module(module_name)
    print(f"{module_name}: OK")

rows = list(csv.DictReader((PROJECT_DIR / "metadata_wav.csv").open("r", encoding="utf-8", newline="")))
missing_audio = [
    str(p) for row in rows
    if not (p := (PROJECT_DIR / row["audio_path"]).resolve()).exists()
]
print(f"Metadata rows: {len(rows)}")
if missing_audio:
    for p in missing_audio[:10]:
        print(p)
    raise FileNotFoundError(f"{len(missing_audio)} audio files missing from metadata.")
print("Preflight checks passed.")

In [ ]:
# Use the official MMS Hausa config.json as --base-config so the model architecture
# used for fine-tuning exactly matches the pretrained checkpoint weights.
prepare_cmd = [
    "python", "scripts/prepare_mms_vits_dataset.py",
    "--metadata-csv", "metadata_wav.csv",
    "--audio-root", ".",
    "--output-dir", PREP_DIR.name,
    "--base-config", MMS_DIR / "config.json",
    "--pronunciation-overrides", "configs/pronunciation_overrides.tsv",
    "--val-ratio", str(VAL_RATIO),
    "--batch-size", str(TRAIN_BATCH_SIZE),
    "--learning-rate", str(LEARNING_RATE),
    "--max-steps", str(TRAINING_MAX_STEPS),
]
run(prepare_cmd, cwd=PROJECT_DIR)
print(f"Prepared dataset written to {PREP_DIR}")

In [ ]:
manifest = json.loads((PREP_DIR / "manifest.json").read_text(encoding="utf-8"))
prepared_config = json.loads((PREP_DIR / "finetune_config.json").read_text(encoding="utf-8"))

print(json.dumps({
    "num_samples": manifest["num_samples"],
    "num_train": manifest["num_train"],
    "num_val": manifest["num_val"],
    "sample_rate": manifest["sample_rate"],
    "vocab_path": manifest["vocab_path"],
    "pronunciation_overrides": manifest["pronunciation_overrides"],
}, indent=2, ensure_ascii=False))

OUTPUT_DIR = DRIVE_RUNS_DIR if USE_DRIVE_FOR_OUTPUTS else Path(prepared_config["train"]["output_dir"])
print(f"Training outputs will be saved to: {OUTPUT_DIR}")

## Training notes

- `TRAINING_MAX_STEPS = 1500` is a safe starting point for a first adaptation pass.
- `TRAIN_BATCH_SIZE = 4` is a practical default for Colab T4 GPUs.
- `TRAINABLE_MODULES = "dec,dp"` freezes the text encoder and flow network. Only the decoder (voice timbre) and duration predictor (pacing) are trained. This is the correct strategy for tiny datasets — it prevents catastrophic forgetting of Hausa phonetics while adapting the voice.
- If the voice starts to overfit or sound unstable, reduce `TRAINING_MAX_STEPS`.
- For better pronunciation of specific words, add `source|target` entries to `configs/pronunciation_overrides.tsv`, then rerun the prepare and train cells.
- To fine-tune the full model (only safe with hours of data), set `TRAINABLE_MODULES = "all"`.

In [ ]:
train_cmd = [
    "python", "scripts/finetune_mms_vits.py",
    "--config", PREP_DIR / "finetune_config.json",
    "--vits-dir", VITS_DIR,
    "--pretrained-dir", MMS_DIR,
    "--output-dir", OUTPUT_DIR,
    "--trainable-modules", TRAINABLE_MODULES,
]
run(train_cmd, cwd=PROJECT_DIR)

In [ ]:
%load_ext tensorboard
%tensorboard --logdir {OUTPUT_DIR}

In [ ]:
sample_out = OUTPUT_DIR / "samples" / "test_sentence.wav"

synth_cmd = [
    "python", "scripts/synthesize_mms_vits.py",
    "--config", PREP_DIR / "finetune_config.json",
    "--vits-dir", VITS_DIR,
    "--checkpoint-dir", OUTPUT_DIR,
    "--text", TEST_SENTENCE,
    "--out-wav", sample_out,
    "--pronunciation-overrides", PROJECT_DIR / "configs" / "pronunciation_overrides.tsv",
]
run(synth_cmd, cwd=PROJECT_DIR)

from IPython.display import Audio, display
display(Audio(str(sample_out), rate=16000))

In [ ]:
archive_path = Path("/content/mms_hau_finetune_outputs.zip")
if archive_path.exists():
    archive_path.unlink()
run(["zip", "-r", archive_path, OUTPUT_DIR])
print(f"Saved archive to {archive_path}")